# 1 · Set Up the FABRIC Slice

Provision a 3-node slice (Alice / BMv2 switch / Bob), upload QFabric, install BMv2,
compile the P4 quantum-channel program, start the switch, and wire the data-plane network.

This notebook is a thin wrapper over the tested functions in `scripts/deploy_fabric.py`
(`create_slice`, `upload_project`, `install_deps`, `configure_switch`, `setup_dataplane_ips`),
so it stays in lock-step with the command-line deployer.

**Prerequisite:** `00_overview` (env check passed, FABRIC tokens configured).
After this notebook the slice is live and the data plane is running — go to `02_run_experiment`.

### At a glance
- **Purpose:** stand up the 3-node slice and bring the quantum data plane online.
- **Inputs:** `SLICE_NAME`, the three FABRIC sites, and `SCENARIO` (sets the initial loss threshold).
- **Outputs:** a live slice — BMv2 running on the switch with the fiber-loss + MAC-rewrite tables, and data-plane IPs (10.10.1.x) on Alice/Bob for the classical channel.
- **Runs on / runtime:** FABRIC JupyterHub; **~10–20 min** (slice provisioning + first-time BMv2/p4c build on the switch).
- **If something fails:** BMv2 build errors are almost always a missing apt dep on the switch — the error names it; add it to `scripts/install_bmv2.sh`. Re-running this notebook is safe (it reuses an existing slice and preserves the node venvs).

## Configuration

In [ ]:
SLICE_NAME = 'qfabric-bb84-2'
SITE_ALICE = 'TACC'      # Alice site
SITE_BOB   = 'STAR'      # Bob site (different site → real WAN classical channel)
SITE_SW    = 'TACC'      # BMv2 switch (colocated with Alice)
SCENARIO   = 'validation/scenarios/fabric_1km.yml'   # repo-relative; sizes the loss model

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

# deploy_fabric.py holds the tested provisioning/run logic; the notebooks are
# thin wrappers around it so there is a single source of truth.
import deploy_fabric as df
from qne.config import ScenarioConfig

fablib = df.get_fablib()

## Step 1 — Provision the slice (Alice, Bob, switch + L2 networks)
Submits the slice and waits for SSH. Takes a few minutes.

In [ ]:
# Reuse the slice if it already exists (re-runnable); otherwise provision it.
try:
    slice_obj = fablib.get_slice(name=SLICE_NAME)
    print(f"Reusing existing slice '{SLICE_NAME}' (state: {slice_obj.get_state()})")
except Exception:
    slice_obj = df.create_slice(fablib, SLICE_NAME, SITE_ALICE, SITE_BOB, SITE_SW)

## Step 2 — Upload QFabric and install dependencies
`upload_project` copies only the needed files (no `.venv`/`.git`). `install_deps` ensures
pip/venv on Alice & Bob and builds BMv2 on the switch (several minutes on first run).

In [ ]:
df.upload_project(slice_obj)
df.install_deps(slice_obj)

## Step 3 — Compile P4, start the switch, and set up the data plane
Computes the fiber-loss threshold from the scenario, starts BMv2 (durable `systemd-run`)
with the loss-model + MAC-rewrite tables, and assigns data-plane IPs so the classical
channel rides the FABRIC L2 link (the management network blocks cross-site TCP).

In [ ]:
threshold = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO).loss_threshold_u32
print(f'P4 loss threshold (u32): {threshold}')

alice_mac, bob_mac, sw_alice_mac, sw_bob_mac, _, _ = df.configure_switch(slice_obj, threshold)
alice_ip, bob_ip = df.setup_dataplane_ips(slice_obj, alice_mac, bob_mac)
print('\nSlice ready: switch running, data plane up.')

---
**Next:** `02_run_experiment`.